In [17]:
!pip install -q mediapipe opencv-python-headless numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 10.0 MB/s eta 0:00:00


In [18]:
import os
import time
import csv
import base64
import urllib.request
from datetime import datetime

import cv2
import numpy as np
import mediapipe as mp

from IPython.display import display, Javascript, HTML, Audio
from google.colab.output import eval_js
from google.colab import output

MODEL_PATH = "/content/face_landmarker.task"

MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_landmarker/face_landmarker/float16/1/"
    "face_landmarker.task"
)

if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

print("Libraries imported successfully.")
print("MediaPipe model downloaded:", os.path.exists(MODEL_PATH))

Libraries imported successfully.
MediaPipe model downloaded: True


In [19]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(
    model_asset_path=MODEL_PATH
)

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1,
    output_face_blendshapes=True
)

landmarker = vision.FaceLandmarker.create_from_options(options)

print("Face Landmarker is ready.")

Face Landmarker is ready.


In [21]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

camera_javascript = Javascript("""
async function startDriveSafeCamera() {
  if (window.driveSafeStream) {
    window.driveSafeStream.getTracks().forEach(t => t.stop());
  }

  const stream = await navigator.mediaDevices.getUserMedia({
    video: { width: 640, height: 480 },
    audio: false
  });

  window.driveSafeStream = stream;

  let video = document.getElementById('drivesafe-video');
  if (!video) {
    video = document.createElement('video');
    video.id = 'drivesafe-video';
    video.autoplay = true;
    video.playsInline = true;
    video.style.width = '480px';
    video.style.maxWidth = '100%';
    video.style.borderRadius = '12px';
    video.style.display = 'block';
    video.style.margin = '10px 0';
    document.body.appendChild(video);
  }

  video.srcObject = stream;
  await video.play();
  return 'Camera started';
}

async function driveSafeCaptureFrame() {
  const video = document.getElementById('drivesafe-video');
  if (!video || !video.videoWidth) {
    throw new Error('Camera is not ready. Start the camera first.');
  }

  const canvas = document.createElement('canvas');
  canvas.width = video.videoWidth;
  canvas.height = video.videoHeight;
  canvas.getContext('2d').drawImage(video, 0, 0);

  return canvas.toDataURL('image/jpeg', 0.8);
}

function stopDriveSafeCamera() {
  if (window.driveSafeStream) {
    window.driveSafeStream.getTracks().forEach(t => t.stop());
    window.driveSafeStream = null;
  }
  const video = document.getElementById('drivesafe-video');
  if (video) video.srcObject = null;
  return 'Camera stopped';
}
""")

display(camera_javascript)
print(eval_js("startDriveSafeCamera()"))

<IPython.core.display.Javascript object>

Camera started


In [22]:
def capture_frame():
    data_url = eval_js("driveSafeCaptureFrame()")
    image_bytes = b64decode(data_url.split(",")[1])

    frame = cv2.imdecode(
        np.frombuffer(image_bytes, dtype=np.uint8),
        cv2.IMREAD_COLOR
    )

    if frame is None:
        raise ValueError("Could not decode the camera frame.")

    return frame


def detect_eye_state(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    timestamp_ms = int(time.monotonic() * 1000)

    result = landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    if not result.face_landmarks or not result.face_blendshapes:
        return frame, None, "FACE NOT DETECTED"

    scores = {
        category.category_name: category.score
        for category in result.face_blendshapes[0].categories
    }

    left = scores.get("eyeBlinkLeft", 0.0)
    right = scores.get("eyeBlinkRight", 0.0)

    # Approximate eye-closure signal:
    # both eye-blink scores must exceed the threshold.
    closed = left >= 0.55 and right >= 0.55

    label = "EYES CLOSED SIGNAL" if closed else "EYES OPEN SIGNAL"

    cv2.putText(
        frame,
        label,
        (15, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255) if closed else (0, 200, 0),
        2
    )

    cv2.putText(
        frame,
        f"Left: {left:.2f}  Right: {right:.2f}",
        (15, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )

    return frame, closed, label


print("Detection functions created successfully.")

Detection functions created successfully.


In [23]:
import pandas as pd
from IPython.display import Audio, display

THRESHOLD_SECONDS = 1.5
events = []

def record_event(event_type, message):
    events.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "event": event_type,
        "message": message
    })

    print(
        f"[{events[-1]['timestamp']}] "
        f"{event_type}: {message}"
    )


def play_alarm():
    # Generate a short alarm tone in the notebook.
    sample_rate = 22050
    duration = 0.45
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

    tone = 0.5 * np.sin(2 * np.pi * 880 * t)
    envelope = np.minimum(t / 0.03, 1.0)
    envelope *= np.minimum((duration - t) / 0.03, 1.0)

    display(Audio(tone * envelope, rate=sample_rate, autoplay=True))


print("Alarm and event logging are ready.")

Alarm and event logging are ready.


In [24]:
from PIL import Image as PILImage
from IPython.display import display

THRESHOLD_SECONDS = 1.5

closed_start = None
warning_active = False
alert_count = 0

session_start = time.time()
max_runtime_seconds = 300

image_output = None

print("DriveSafe AI monitoring started.")
print("Use Interrupt execution to stop monitoring.")

try:
    while time.time() - session_start < max_runtime_seconds:
        frame = capture_frame()
        annotated, closed, label = detect_eye_state(frame)

        now = time.time()

        if closed is True:
            if closed_start is None:
                closed_start = now

            closed_duration = now - closed_start

            cv2.putText(
                annotated,
                f"Closure signal: {closed_duration:.1f}s",
                (15, 95),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2
            )

            if (
                closed_duration >= THRESHOLD_SECONDS
                and not warning_active
            ):
                warning_active = True
                alert_count += 1

                record_event(
                    "WARNING",
                    "Sustained eye-closure signal detected"
                )

                print(
                    "WARNING: Stop safely and rest. "
                    "Do not continue driving while sleepy."
                )

                play_alarm()

        else:
            closed_start = None

            if warning_active:
                record_event(
                    "INFO",
                    "Eye-closure signal returned below threshold"
                )

            warning_active = False

        cv2.putText(
            annotated,
            f"Alerts: {alert_count}",
            (15, 125),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 0),
            2
        )

        rgb_frame = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        pil_frame = PILImage.fromarray(rgb_frame)

        if image_output is None:
            image_output = display(pil_frame, display_id=True)
        else:
            image_output.update(pil_frame)

        time.sleep(0.15)

except KeyboardInterrupt:
    print("Monitoring interrupted by user.")

except Exception as error:
    print("Monitoring error:", error)

finally:
    print("Monitoring loop ended.")
    print("Total alerts:", alert_count)
    print("Session duration:", round(time.time() - session_start, 1), "seconds")

DriveSafe AI monitoring started.
Use Interrupt execution to stop monitoring.
Monitoring error: ReferenceError: driveSafeCaptureFrame is not defined
Monitoring loop ended.
Total alerts: 0
Session duration: 0.4 seconds


In [25]:
if events:
    report_df = pd.DataFrame(events)
    display(report_df)
else:
    report_df = pd.DataFrame(
        columns=["timestamp", "event", "message"]
    )
    print("No events were recorded during this session.")

No events were recorded during this session.


In [26]:
from google.colab import files

report_path = "/content/drivesafe_session_report.csv"

report_df.to_csv(report_path, index=False)

files.download(report_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
# Re-inject JavaScript definition in case the browser session context was lost
from IPython.display import display, Javascript
from google.colab.output import eval_js

camera_javascript = Javascript("""
async function startDriveSafeCamera() {
  if (window.driveSafeStream) {
    window.driveSafeStream.getTracks().forEach(t => t.stop());
  }

  const stream = await navigator.mediaDevices.getUserMedia({
    video: { width: 640, height: 480 },
    audio: false
  });

  window.driveSafeStream = stream;

  let video = document.getElementById('drivesafe-video');
  if (!video) {
    video = document.createElement('video');
    video.id = 'drivesafe-video';
    video.autoplay = true;
    video.playsInline = true;
    video.style.width = '480px';
    video.style.maxWidth = '100%';
    video.style.borderRadius = '12px';
    video.style.display = 'block';
    video.style.margin = '10px 0';
    document.body.appendChild(video);
  }

  video.srcObject = stream;
  await video.play();
  return 'Camera started';
}

async function driveSafeCaptureFrame() {
  const video = document.getElementById('drivesafe-video');
  if (!video || !video.videoWidth) {
    throw new Error('Camera is not ready. Start the camera first.');
  }

  const canvas = document.createElement('canvas');
  canvas.width = video.videoWidth;
  canvas.height = video.videoHeight;
  canvas.getContext('2d').drawImage(video, 0, 0);

  return canvas.toDataURL('image/jpeg', 0.8);
}

function stopDriveSafeCamera() {
  if (window.driveSafeStream) {
    window.driveSafeStream.getTracks().forEach(t => t.stop());
    window.driveSafeStream = null;
  }
  const video = document.getElementById('drivesafe-video');
  if (video) video.srcObject = null;
  return 'Camera stopped';
}
""")

display(camera_javascript)
try:
    print(eval_js("stopDriveSafeCamera()"))
except Exception as e:
    print("Error stopping camera:", e)

<IPython.core.display.Javascript object>

Camera stopped


In [28]:
!pip -q install flask pyngrok

In [29]:
%%writefile /content/app.py
from flask import Flask, Response
app = Flask(__name__)

PAGE = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>DriveSafe AI | Driver Safety</title>
<style>
:root{--bg:#08111f;--panel:#122139;--line:#293d55;--text:#edf5ff;--muted:#9aacc1;--cyan:#42d7e8;--green:#37d99a;--yellow:#ffca62;--red:#ff647c}
*{box-sizing:border-box}body{margin:0;background:radial-gradient(ellipse at 10% 0%,#19324e,transparent 42%),var(--bg);color:var(--text);font:15px/1.5 Arial, sans-serif}
header{padding:17px 4%;display:flex;justify-content:space-between;align-items:center;border-bottom:1px solid var(--line);background:#08111fe8;position:sticky;top:0}.brand{display:flex;align-items:center;gap:12px}.logo{background:linear-gradient(135deg,var(--cyan),#4b8cf5);color:#06111f;border-radius:12px;padding:8px 12px;font-weight:900;font-size:20px}.brand h1{margin:0;font-size:18px}.brand p{margin:0;color:var(--muted);font-size:12px}.pill,.tag{border:1px solid var(--line);border-radius:999px;padding:6px 10px;color:var(--muted);font-size:12px}
main{max-width:1300px;margin:auto;padding:27px 3%}.intro{display:flex;justify-content:space-between;align-items:end;gap:15px;margin-bottom:22px}.eyebrow{color:var(--cyan);font-weight:bold;font-size:11px;letter-spacing:2px}.intro h2{font-size:clamp(27px,3vw,38px);margin:6px 0}.intro p{color:var(--muted);margin:0;max-width:720px}.clock{text-align:right;color:var(--muted);white-space:nowrap}.clock strong{display:block;color:var(--text);font-size:18px}
.grid{display:grid;grid-template-columns:1.5fr .8fr;gap:18px}.panel{background:linear-gradient(145deg,#13233a,#0e1929);border:1px solid var(--line);border-radius:18px;overflow:hidden}.head{padding:16px 19px;border-bottom:1px solid var(--line);display:flex;justify-content:space-between;align-items:center;gap:10px}.head h3{font-size:14px;margin:0}.sub,.note{color:var(--muted);font-size:12px}.cam{aspect-ratio:16/9;min-height:250px;background:#030912;display:grid;place-items:center;position:relative;overflow:hidden}.cam video{width:100%;height:100%;object-fit:cover;transform:scaleX(-1);display:none}.cam video.on{display:block}.placeholder{text-align:center;max-width:390px;padding:22px}.placeholder b{display:block;font-size:18px;margin:8px}.placeholder p{font-size:13px;color:var(--muted)}.overlay{position:absolute;top:12px;left:12px;right:12px;display:flex;justify-content:space-between;pointer-events:none}.btns{display:flex;gap:9px;flex-wrap:wrap;padding:15px 18px}.btn{border:0;border-radius:10px;padding:11px 15px;font-weight:bold;cursor:pointer}.primary{background:linear-gradient(135deg,var(--cyan),#75a4ff);color:#06111f}.secondary{background:#1a2d44;color:var(--text);border:1px solid #344a63}.btn:disabled{opacity:.45}.status{padding:20px}.label{font-size:11px;color:var(--muted);letter-spacing:1.4px;text-transform:uppercase}.state{display:flex;align-items:center;gap:12px;margin:13px 0}.icon{width:46px;height:46px;border-radius:14px;background:#173b35;color:var(--green);display:grid;place-items:center;font-size:23px}.state h3{margin:0;font-size:21px}.state p{margin:3px 0 0;color:var(--muted);font-size:12px}.warn .icon{background:#3a2d17;color:var(--yellow)}.alert .icon{background:#421e2a;color:var(--red)}.alert h3{color:var(--red)}hr{border:0;border-top:1px solid var(--line);margin:18px 0}.metric{display:flex;justify-content:space-between;gap:10px;margin:13px 0}.metric span{font-size:12px;color:var(--muted)}.metric strong{font-size:14px}.meter{height:7px;background:#293b50;border-radius:8px;overflow:hidden}.meter div{height:100%;width:0;background:var(--red)}.alarmrow{display:flex;align-items:center;justify-content:space-between}.switch{width:45px;height:25px;border:0;border-radius:99px;background:#30445b;padding:3px;cursor:pointer}.switch span{display:block;width:19px;height:19px;border-radius:50%;background:white}.switch.on{background:#2abf8b}.switch.on span{transform:translateX(20px)}
.bottom{display:grid;grid-template-columns:1fr 1fr;gap:18px;margin-top:18px}.settings{padding:19px}.setting{display:flex;justify-content:space-between;gap:12px;margin:15px 0 7px;font-size:13px;font-weight:bold}.setting small{display:block;color:var(--muted);font-weight:normal}.setting output{color:var(--cyan)}input[type=range]{width:100%;accent-color:var(--cyan)}.historytop{padding:17px 19px;display:flex;align-items:center;justify-content:space-between;border-bottom:1px solid var(--line)}.historytop h3{font-size:14px;margin:0}.events{padding:5px 19px;max-height:240px;overflow:auto}.event{display:grid;grid-template-columns:8px 1fr auto;gap:10px;padding:12px 0;border-bottom:1px solid #20334a;font-size:12px}.event strong{display:block}.event small,.event time{color:var(--muted)}.bullet{width:8px;height:8px;border-radius:50%;background:var(--yellow);margin-top:5px}.empty{text-align:center;padding:24px 0;color:var(--muted);font-size:12px}.how{padding:19px;margin-top:18px}.how h3{font-size:14px;margin:0 0 12px}.steps{display:grid;grid-template-columns:repeat(3,1fr);gap:12px}.step{padding:13px;border:1px solid var(--line);border-radius:12px;background:#0c1929}.step b{font-size:12px}.step p{color:var(--muted);font-size:12px;margin:5px 0 0}.notice{display:none;color:#ffb7c1;background:#301923;border:1px solid #75404b;border-radius:10px;padding:11px 14px;margin:0 18px 14px;font-size:12px}.notice.show{display:block}.dot{display:inline-block;width:7px;height:7px;background:#60748c;border-radius:50%;margin-right:5px}.dot.live{background:var(--green)}footer{text-align:center;padding:22px;color:#71869f;font-size:11px}
@media(max-width:850px){.grid,.bottom{grid-template-columns:1fr}.intro{display:block}.clock{text-align:left;margin-top:12px}.steps{grid-template-columns:1fr}}@media(max-width:480px){header{padding:13px}.pill{display:none}.cam{min-height:210px}.btn{flex:1}}
</style></head><body>
<header><div class="brand"><div class="logo">DS</div><div><h1>DriveSafe AI</h1><p>Driver Drowsiness Detection System</p></div></div><div class="pill"><span class="dot" id="hdot"></span><span id="hstatus">System idle</span></div></header>
<main><section class="intro"><div><div class="eyebrow">AI-POWERED DRIVER SAFETY</div><h2>Stay alert. Arrive safe.</h2><p>Monitor eye-blink signals from your camera, trigger an audible warning for prolonged eye closure, and export a session event report.</p></div><div class="clock"><strong id="time">--:--:--</strong><span id="date"></span></div></section>
<section class="grid"><article class="panel"><div class="head"><div><h3>Live camera monitoring</h3><div class="sub">Face and eye analysis runs in your browser</div></div><span class="tag" id="camlabel">Camera off</span></div><div class="cam"><video id="video" autoplay muted playsinline></video><div class="placeholder" id="placeholder"><div style="font-size:45px">◉</div><b>Camera is ready</b><p>Click Start monitoring and allow camera permission. Keep your face visible and use good lighting.</p></div><div class="overlay"><span class="tag" id="livetag">● STANDBY</span><span class="tag" id="facetag">Face: unknown</span></div></div><div class="notice" id="notice"></div><div class="btns"><button class="btn primary" id="start">▶ Start monitoring</button><button class="btn secondary" id="stop" disabled>■ Stop</button><button class="btn secondary" id="test">♫ Test alarm</button></div></article>
<aside class="panel status"><div class="label">Current driver state</div><div class="state" id="state"><div class="icon" id="sicon">✓</div><div><h3 id="stitle">Ready to monitor</h3><p id="stext">Start the camera to begin</p></div></div><hr><div class="metric"><span>Eye state</span><strong id="eye">Unknown</strong></div><div class="metric"><span>Eye-closure duration</span><strong id="closed">0.0 s</strong></div><div class="meter"><div id="bar"></div></div><div class="metric"><span>Monitoring time</span><strong id="elapsed">00:00</strong></div><div class="metric"><span>Warnings this session</span><strong id="warnings">0</strong></div><hr><div class="alarmrow"><div><b style="font-size:13px">Audible alarm</b><div class="sub">Sound when warning is triggered</div></div><button class="switch on" id="alarm"><span></span></button></div><p class="note">Prototype only; not a certified safety system. Do not test while driving.</p></aside></section>
<section class="bottom"><article class="panel settings"><div class="head" style="padding:0 0 15px;border:0"><div><h3>Detection settings</h3><div class="sub">Adjust warning sensitivity</div></div><span class="tag">CUSTOM</span></div><div class="setting"><div>Closed-eye alert delay<small>Duration before warning</small></div><output id="dout">1.5 sec</output></div><input type="range" id="delay" min=".8" max="3.5" step=".1" value="1.5"><div class="setting" style="margin-top:22px"><div>Eye-closure sensitivity<small>Higher means stronger blink signal needed</small></div><output id="sout">0.55</output></div><input type="range" id="sensitivity" min=".35" max=".8" step=".05" value=".55"><p class="note">Never change settings while driving. Camera quality and lighting affect detection.</p></article>
<article class="panel"><div class="historytop"><div><h3>Session event history</h3><div class="sub">Events recorded in this session</div></div><button class="btn secondary" id="export" style="padding:8px 10px">↓ Export CSV</button></div><div class="events" id="events"><div class="empty">No events yet. Start monitoring to record events.</div></div></article></section>
<section class="panel how"><h3>How DriveSafe AI works</h3><div class="steps"><div class="step"><b>01 · Face and eye analysis</b><p>A browser-based MediaPipe model estimates eye-blink signals from camera frames.</p></div><div class="step"><b>02 · Sustained closure check</b><p>The system checks whether both eyes remain closed longer than your selected delay.</p></div><div class="step"><b>03 · Alarm and event log</b><p>A warning sound can play and a timestamped event is added to the session history.</p></div></div></section><footer>DriveSafe AI · Colab-hosted web prototype · Never rely on this alone for driving safety.</footer></main>
<script type="module">
const $=id=>document.getElementById(id);let stream=null,model=null,running=false,raf=0,lastFrame=-1,closedAt=null,alertOn=false,alarmOn=true,audio=null,alarmLoop=null,startAt=0,count=0,events=[],delay=1.5,sensitivity=.55;
const video=$('video'),version='0.10.14',modelURL='https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task';
function notice(s){$('notice').textContent=s;$('notice').classList.add('show')}function clearNotice(){$('notice').textContent='';$('notice').classList.remove('show')}
function tick(){let d=new Date();$('time').textContent=d.toLocaleTimeString();$('date').textContent=d.toLocaleDateString(undefined,{weekday:'short',year:'numeric',month:'short',day:'numeric'});if(running){let n=Math.floor((Date.now()-startAt)/1000);$('elapsed').textContent=String(Math.floor(n/60)).padStart(2,'0')+':'+String(n%60).padStart(2,'0')}}setInterval(tick,500);tick();
function state(k,t,s){$('state').className='state '+(k==='alert'?'alert':k==='warn'?'warn':'');$('sicon').textContent=k==='alert'?'!':k==='warn'?'◷':'✓';$('stitle').textContent=t;$('stext').textContent=s}
function log(type,detail){events.unshift({time:new Date().toISOString(),type,detail});render()}
function esc(s){return String(s).replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]))}
function render(){$('events').innerHTML=events.length?events.slice(0,40).map(e=>`<div class="event"><span class="bullet" style="background:${e.type==='DROWSINESS WARNING'?'var(--red)':e.type==='MONITORING STARTED'?'var(--green)':'var(--yellow)'}"></span><div><strong>${esc(e.type)}</strong><small>${esc(e.detail)}</small></div><time>${new Date(e.time).toLocaleTimeString()}</time></div>`).join(''):'<div class="empty">No events yet. Start monitoring to record events.</div>'}
function tone(){if(!alarmOn||!running)return;try{audio=audio||new(window.AudioContext||window.webkitAudioContext)();if(audio.state==='suspended')audio.resume();let o=audio.createOscillator(),g=audio.createGain();o.type='sawtooth';o.frequency.setValueAtTime(880,audio.currentTime);o.frequency.setValueAtTime(660,audio.currentTime+.2);g.gain.setValueAtTime(.001,audio.currentTime);g.gain.exponentialRampToValueAtTime(.16,audio.currentTime+.03);g.gain.setValueAtTime(.16,audio.currentTime+.3);g.gain.exponentialRampToValueAtTime(.001,audio.currentTime+.4);o.connect(g);g.connect(audio.destination);o.start();o.stop(audio.currentTime+.42)}catch(e){}}
function startAlarm(){if(alarmOn&&!alarmLoop){tone();alarmLoop=setInterval(tone,850)}}function stopAlarm(){if(alarmLoop){clearInterval(alarmLoop);alarmLoop=null}}
function reset(){closedAt=null;alertOn=false;stopAlarm();$('closed').textContent='0.0 s';$('bar').style.width='0%'}
function handle(res,now){let faces=res.faceBlendshapes||[];if(!faces.length){$('facetag').textContent='Face: not detected';$('eye').textContent='No face';if(closedAt!==null)reset();state('warn','Face not detected','Move into the camera view');return}
$('facetag').textContent='Face: detected';let c=faces[0].categories||[],l=c.find(x=>x.categoryName==='eyeBlinkLeft')?.score||0,r=c.find(x=>x.categoryName==='eyeBlinkRight')?.score||0,shut=l>=sensitivity&&r>=sensitivity;$('eye').textContent=shut?'Closed':'Open';
if(shut){if(closedAt===null)closedAt=now;let dur=(now-closedAt)/1000;$('closed').textContent=dur.toFixed(1)+' s';$('bar').style.width=Math.min(100,dur/delay*100)+'%';if(dur>=delay){if(!alertOn){alertOn=true;count++;$('warnings').textContent=count;log('DROWSINESS WARNING',`Eyes appear closed for ${dur.toFixed(1)} seconds`)}state('alert','Drowsiness warning','Open your eyes and take a safe break');startAlarm()}else state('warn','Eyes closed','Checking duration before alert')}
else{if(alertOn)log('WARNING CLEARED','Eye state returned to open');reset();state('normal','Alert / awake','Eye state appears normal')}}
async function loop(){if(!running)return;try{if(video.readyState>=2&&video.currentTime!==lastFrame){lastFrame=video.currentTime;let now=performance.now();handle(model.detectForVideo(video,now),now)}}catch(e){console.error(e);notice('Detection error: '+e.message)}raf=requestAnimationFrame(loop)}
async function start(){if(running)return;clearNotice();$('start').disabled=true;try{if(!navigator.mediaDevices?.getUserMedia)throw Error('Camera requires HTTPS or localhost.');state('warn','Loading AI model','Please allow camera access');let v=await import(`https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@${version}/vision_bundle.mjs`);let fs=await v.FilesetResolver.forVisionTasks(`https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@${version}/wasm`);model=await v.FaceLandmarker.createFromOptions(fs,{baseOptions:{modelAssetPath:modelURL,delegate:'GPU'},runningMode:'VIDEO',numFaces:1,outputFaceBlendshapes:true});stream=await navigator.mediaDevices.getUserMedia({video:{facingMode:'user',width:{ideal:1280},height:{ideal:720}},audio:false});video.srcObject=stream;await video.play();video.classList.add('on');$('placeholder').style.display='none';running=true;startAt=Date.now();lastFrame=-1;count=0;$('warnings').textContent='0';reset();$('hdot').classList.add('live');$('hstatus').textContent='Monitoring active';$('camlabel').textContent='Camera active';$('livetag').textContent='● LIVE MONITORING';$('start').disabled=true;$('stop').disabled=false;log('MONITORING STARTED','Camera monitoring session started');state('normal','Monitoring active','Keep your face visible to the camera');loop()}catch(e){console.error(e);state('warn','Could not start','Check camera permission and internet');notice('Unable to start: '+e.message+' Use Chrome/Edge, allow camera access, and open the HTTPS website.');if(stream)stream.getTracks().forEach(t=>t.stop());stream=null;$('start').disabled=false}}
function stop(){if(!running&&!stream)return;running=false;cancelAnimationFrame(raf);stopAlarm();if(stream)stream.getTracks().forEach(t=>t.stop());stream=null;video.srcObject=null;video.classList.remove('on');$('placeholder').style.display='block';$('placeholder').innerHTML='<div style="font-size:45px">◉</div><b>Monitoring stopped</b><p>Start again whenever you are ready.</p>';$('hdot').classList.remove('live');$('hstatus').textContent='System idle';$('camlabel').textContent='Camera off';$('livetag').textContent='● STANDBY';$('start').disabled=false;$('stop').disabled=true;reset();state('normal','Monitoring stopped','Session data is ready to export');log('MONITORING STOPPED','Camera monitoring session stopped')}
$('start').onclick=start;$('stop').onclick=stop;$('test').onclick=()=>{try{audio=audio||new(window.AudioContext||window.webkitAudioContext)();if(audio.state==='suspended')audio.resume();let o=audio.createOscillator(),g=audio.createGain();o.frequency.value=880;g.gain.value=.15;o.connect(g);g.connect(audio.destination);o.start();o.stop(audio.currentTime+.25);log('ALARM TEST','Manual alarm test played')}catch(e){notice('Audio playback was blocked. Click the page and try again.')}};
$('alarm').onclick=()=>{alarmOn=!alarmOn;$('alarm').classList.toggle('on',alarmOn);if(!alarmOn)stopAlarm();else if(alertOn&&running)startAlarm();log('ALARM SETTING',alarmOn?'Audible alarm enabled':'Audible alarm disabled')};
$('delay').oninput=e=>{delay=+e.target.value;$('dout').textContent=delay.toFixed(1)+' sec'};$('sensitivity').oninput=e=>{sensitivity=+e.target.value;$('sout').textContent=sensitivity.toFixed(2)};
$('export').onclick=()=>{let rows=[['timestamp','event','details'],...events.slice().reverse().map(e=>[e.time,e.type,e.detail])],csv=rows.map(r=>r.map(v=>'"'+String(v).replace(/"/g,'""')+'"').join(',')).join('\r\n'),url=URL.createObjectURL(new Blob([csv],{type:'text/csv;charset=utf-8'})),a=document.createElement('a');a.href=url;a.download='drivesafe_session_report.csv';a.click();URL.revokeObjectURL(url)};
addEventListener('beforeunload',()=>{stopAlarm();if(stream)stream.getTracks().forEach(t=>t.stop())});
</script></body></html>"""

@app.route("/")
def index():
    return Response(PAGE, mimetype="text/html")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8501, debug=False, use_reloader=False)


Overwriting /content/app.py


In [30]:
import getpass, subprocess, time, os
from pyngrok import ngrok

# Paste your ngrok authtoken when prompted. Get it from:
# https://dashboard.ngrok.com/get-started/your-authtoken
token = getpass.getpass("Paste your ngrok authtoken: ").strip()
ngrok.set_auth_token(token)

# Clear previous server/tunnels if this cell is rerun
try:
    ngrok.kill()
except Exception:
    pass

app_process = subprocess.Popen(
    ["python", "/content/app.py"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)
time.sleep(3)

public_url = ngrok.connect(8501, "http").public_url
print("✅ DriveSafe AI website is starting.")
print("🌐 Open this website link:", public_url)
print("Keep this Colab cell/runtime running while using the website.")

Paste your ngrok authtoken: ··········
✅ DriveSafe AI website is starting.
🌐 Open this website link: https://entire-showbiz-deputize.ngrok-free.dev
Keep this Colab cell/runtime running while using the website.


In [31]:
from pyngrok import ngrok
import requests

# Try to get active tunnels via the local ngrok API client (port 4040)
try:
    response = requests.get("http://localhost:4040/api/tunnels")
    if response.status_code == 200:
        tunnels = response.json().get("tunnels", [])
        print("Checking active ngrok tunnels...")
        for tunnel in tunnels:
            print(f"Active Tunnel: {tunnel['public_url']} -> {tunnel['config']['addr']}")
    else:
        print("No local API detected on port 4040.")
except Exception as e:
    print("Could not query local API directly:", e)
    print("You may need to run the setup cell above (y4cceC8jBGTG) again to establish your session.")

Checking active ngrok tunnels...
Active Tunnel: https://entire-showbiz-deputize.ngrok-free.dev -> http://localhost:8501


In [32]:
!pip install -q pyngrok
from getpass import getpass
from pyngrok import ngrok

token = getpass("Paste your ngrok authtoken: ")
ngrok.set_auth_token(token)

tunnel = ngrok.connect(8501, "http")
print("Open your website here:", tunnel.public_url)

Paste your ngrok authtoken: ··········
Open your website here: https://entire-showbiz-deputize.ngrok-free.dev


In [33]:
!pip -q install flask pyngrok

In [34]:
import requests

try:
    response = requests.get("http://127.0.0.1:8501", timeout=5)
    print("Server is running. Status:", response.status_code)
except Exception as e:
    print("Server is NOT running:", e)

Server is running. Status: 200


In [35]:
import subprocess
import time
import requests
import os

# Stop an older app.py process, if present
subprocess.run(
    ["bash", "-lc", "pkill -f 'python.*app.py' || true"],
    capture_output=True
)

# Start the website server
log_file = open("/content/drivesafe_flask.log", "w")
process = subprocess.Popen(
    ["python", "-u", "/content/app.py"],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Wait for Flask to respond
for attempt in range(20):
    try:
        response = requests.get(
            "http://127.0.0.1:8501", timeout=2
        )
        print("Website server started!")
        print("HTTP status:", response.status_code)
        break
    except requests.RequestException:
        time.sleep(1)
else:
    print("Server failed to start. Check the logs:")
    print(open("/content/drivesafe_flask.log").read())

Website server started!
HTTP status: 200


In [36]:
import os

print("Files in /content:")
print(os.listdir("/content"))

Files in /content:
['.config', 'drivesafe_session_report.csv', 'drivesafe.log', 'face_landmarker.task', 'app.py', 'drivesafe_flask.log', 'sample_data']


In [37]:
%%writefile /content/app.py

Overwriting /content/app.py


In [38]:
%%writefile /content/app.py
from flask import Flask

app = Flask(__name__)

@app.route("/")
def home():
    return """
    <html>
      <head><title>DriveSafe AI</title></head>
      <body style="font-family:Arial;text-align:center;padding:50px">
        <h1>DriveSafe AI</h1>
        <p>Website server is running successfully!</p>
      </body>
    </html>
    """

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8501, debug=False)

Overwriting /content/app.py


In [39]:
import subprocess
import time
import requests

log = open("/content/drivesafe.log", "w")

process = subprocess.Popen(
    ["python", "-u", "/content/app.py"],
    stdout=log,
    stderr=subprocess.STDOUT
)

for i in range(15):
    try:
        r = requests.get("http://127.0.0.1:8501", timeout=2)
        print("SUCCESS! Status:", r.status_code)
        print("Page contains DriveSafe AI:", "DriveSafe AI" in r.text)
        break
    except requests.exceptions.RequestException:
        time.sleep(1)
else:
    print(open("/content/drivesafe.log").read())

SUCCESS! Status: 200
Page contains DriveSafe AI: True


In [40]:
from getpass import getpass
from pyngrok import ngrok

ngrok.kill()

token = getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(token)

tunnel = ngrok.connect(
    addr="http://127.0.0.1:8501",
    proto="http"
)

print("Open your DriveSafe AI website:")
print(tunnel.public_url)

Enter your ngrok authtoken: ··········
Open your DriveSafe AI website:
https://entire-showbiz-deputize.ngrok-free.dev


In [41]:
import requests

try:
    r = requests.get("http://127.0.0.1:8501", timeout=5)
    print("Server status:", r.status_code)
    print(r.text[:200])
except Exception as e:
    print("Error:", e)

Server status: 200
<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>DriveSafe AI | Driver Safety</title>
<style>
:root{--bg:#08111f;-


In [42]:
from getpass import getpass
from pyngrok import ngrok

token = getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(token)

tunnel = ngrok.connect(8501, "http")
print("Your DriveSafe AI website link:")
print(tunnel.public_url)

Enter your ngrok authtoken: ··········
Your DriveSafe AI website link:
https://entire-showbiz-deputize.ngrok-free.dev


In [43]:

%%writefile /content/app.py
from flask import Flask, render_template_string

app = Flask(__name__)

PAGE = r"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>DriveSafe AI</title>
<style>
*{box-sizing:border-box}
body{margin:0;background:#0b1020;color:#edf2ff;font-family:Arial,sans-serif}
header{padding:22px 6%;background:#121a30;border-bottom:1px solid #29334c}
header h1{margin:0;color:#63e6be}header p{color:#aab7d0;margin-bottom:0}
main{max-width:1100px;margin:25px auto;padding:0 18px}
.grid{display:grid;grid-template-columns:1.5fr 1fr;gap:20px}
.card{background:#151e34;border:1px solid #2a3753;border-radius:16px;padding:20px;margin-bottom:20px}
h2{font-size:18px;margin-top:0}
.videoBox{position:relative;background:#050811;border-radius:12px;overflow:hidden;min-height:240px}
video{display:block;width:100%;max-height:480px;transform:scaleX(-1)}
#placeholder{padding:90px 10px;text-align:center;color:#aab7d0}
button{background:#63e6be;color:#07131a;border:0;border-radius:9px;padding:12px 16px;font-weight:bold;cursor:pointer;margin:5px 4px 5px 0}
button.stop{background:#ff758f}button:disabled{opacity:.5}
.status{font-size:22px;font-weight:bold;margin:12px 0;color:#63e6be}
.muted{color:#aab7d0;font-size:14px;line-height:1.5}
label{display:block;margin:14px 0 7px}
input[type=range]{width:100%}
.metric{padding:12px;background:#0c1427;border-radius:10px;margin:9px 0}
#events{max-height:200px;overflow:auto;padding-left:20px}
footer{color:#93a1bb;font-size:12px;padding:20px 0}
@media(max-width:760px){.grid{grid-template-columns:1fr}}
</style>
</head>
<body>
<header>
<h1>DriveSafe AI</h1>
<p>Driver Drowsiness Detection &amp; Alert System</p>
</header>
<main>
<div class="grid">
<section>
<div class="card">
<h2>Live camera monitoring</h2>
<div class="videoBox">
<div id="placeholder">Start camera to begin monitoring</div>
<video id="video" autoplay muted playsinline hidden></video>
</div>
<button id="start">Start Camera &amp; Monitoring</button>
<button id="stop" class="stop" disabled>Stop</button>
<p class="muted" id="message">Camera access requires permission in your browser.</p>
</div>
<div class="card">
<h2>Detection events</h2>
<ul id="events"><li>No events yet</li></ul>
<button id="export">Export event log (CSV)</button>
</div>
</section>
<aside>
<div class="card">
<h2>Monitoring status</h2>
<div class="status" id="status">Not monitoring</div>
<div class="metric">Eye closure score: <b id="score">--</b></div>
<div class="metric">Closed-eye duration: <b id="duration">0.0 s</b></div>
<label for="threshold">Eye-closure threshold: <span id="thresholdText">0.65</span></label>
<input id="threshold" type="range" min="0.4" max="0.9" value="0.65" step="0.05">
<label for="seconds">Alarm delay: <span id="secondsText">1.5</span> seconds</label>
<input id="seconds" type="range" min="0.8" max="3" value="1.5" step="0.1">
<label><input id="sound" type="checkbox" checked> Enable alarm sound</label>
<p class="muted">Detection uses estimated eye-blink scores. Lighting, camera position, glasses and individual differences can affect results.</p>
</div>
<div class="card">
<h2>How it works</h2>
<p class="muted">1. Allow camera access.<br>2. The browser estimates eye-blink scores.<br>3. Sustained eye closure triggers a warning and alarm.<br>4. Events can be exported as CSV.</p>
</div>
</aside>
</div>
<footer>Student prototype for demonstration only. Not a certified driver-safety device. Never test while driving.</footer>
</main>
<script type="module">
import { FilesetResolver, FaceLandmarker } from
"https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@0.10.14";

const video=document.getElementById("video");
const placeholder=document.getElementById("placeholder");
const msg=document.getElementById("message");
const statusEl=document.getElementById("status");
const scoreEl=document.getElementById("score");
const durationEl=document.getElementById("duration");
const threshold=document.getElementById("threshold");
const seconds=document.getElementById("seconds");
const eventsEl=document.getElementById("events");
let landmarker=null,stream=null,active=false,closedSince=null,lastAlarm=0;
let audio=null,events=[],lastVideoTime=-1,raf=null;

threshold.oninput=()=>document.getElementById("thresholdText").textContent=threshold.value;
seconds.oninput=()=>document.getElementById("secondsText").textContent=seconds.value;

function addEvent(text){
 const time=new Date();
 events.unshift({time:time.toLocaleString(),event:text});
 eventsEl.innerHTML="";
 events.slice(0,30).forEach(e=>{
   const li=document.createElement("li");
   li.textContent=e.time+" — "+e.event;
   eventsEl.appendChild(li);
 });
}
function alarm(){
 if(!document.getElementById("sound").checked)return;
 const now=Date.now();
 if(now-lastAlarm<1100)return;
 lastAlarm=now;
 try{
   if(!audio)audio=new AudioContext();
   if(audio.state==="suspended")audio.resume();
   const osc=audio.createOscillator(),gain=audio.createGain();
   osc.type="sawtooth";osc.frequency.value=880;
   gain.gain.value=0.18;
   osc.connect(gain);gain.connect(audio.destination);
   osc.start();osc.stop(audio.currentTime+0.35);
 }catch(e){msg.textContent="Alarm audio blocked. Click the page and allow audio."}
}
async function loadModel(){
 msg.textContent="Loading face-detection model...";
 const vision=await FilesetResolver.forVisionTasks(
   "https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@0.10.14/wasm"
 );
 landmarker=await FaceLandmarker.createFromOptions(vision,{
   baseOptions:{
     modelAssetPath:"https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
   },
   runningMode:"VIDEO",numFaces:1,outputFaceBlendshapes:true
 });
}
async function start(){
 document.getElementById("start").disabled=true;
 try{
   if(!landmarker)await loadModel();
   stream=await navigator.mediaDevices.getUserMedia({
     video:{facingMode:"user"},audio:false
   });
   video.srcObject=stream;
   await video.play();
   video.hidden=false;placeholder.hidden=true;
   active=true;closedSince=null;
   statusEl.textContent="Monitoring";
   statusEl.style.color="#63e6be";
   msg.textContent="Camera active. Keep your face visible.";
   document.getElementById("stop").disabled=false;
   if(!audio)audio=new AudioContext();
   if(audio.state==="suspended")await audio.resume();
   loop();
 }catch(e){
   msg.textContent="Could not start: "+e.message+
     ". Check camera permission and internet access.";
   statusEl.textContent="Camera/model error";
   document.getElementById("start").disabled=false;
 }
}
function loop(){
 if(!active)return;
 if(video.readyState>=2&&video.currentTime!==lastVideoTime){
  lastVideoTime=video.currentTime;
  try{
   const result=landmarker.detectForVideo(video,performance.now());
   const shapes=result.faceBlendshapes?.[0]?.categories||[];
   const left=shapes.find(x=>x.categoryName==="eyeBlinkLeft")?.score;
   const right=shapes.find(x=>x.categoryName==="eyeBlinkRight")?.score;
   if(left===undefined||right===undefined){
    scoreEl.textContent="No face";
    statusEl.textContent="Keep face visible";
    closedSince=null;durationEl.textContent="0.0 s";
   }else{
    const score=(left+right)/2;
    scoreEl.textContent=score.toFixed(2);
    if(score>=Number(threshold.value)){
     if(closedSince===null)closedSince=performance.now();
     const elapsed=(performance.now()-closedSince)/1000;
     durationEl.textContent=elapsed.toFixed(1)+" s";
     if(elapsed>=Number(seconds.value)){
      statusEl.textContent="DROWSINESS ALERT!";
      statusEl.style.color="#ff758f";
      if(elapsed>=Number(seconds.value)&&
         (!events.length||Date.now()-lastAlarm>5000)){
        addEvent("Prolonged eye closure detected");
      }
      alarm();
     }else{
      statusEl.textContent="Eyes closed — monitoring";
      statusEl.style.color="#ffd166";
     }
    }else{
     closedSince=null;durationEl.textContent="0.0 s";
     statusEl.textContent="Alert — attentive";
     statusEl.style.color="#63e6be";
    }
   }
  }catch(e){msg.textContent="Detection error: "+e.message;}
 }
 raf=requestAnimationFrame(loop);
}
function stop(){
 active=false;
 if(raf)cancelAnimationFrame(raf);
 if(stream)stream.getTracks().forEach(t=>t.stop());
 stream=null;video.srcObject=null;video.hidden=true;placeholder.hidden=false;
 statusEl.textContent="Not monitoring";statusEl.style.color="#63e6be";
 document.getElementById("start").disabled=false;
 document.getElementById("stop").disabled=true;
 msg.textContent="Monitoring stopped.";
}
document.getElementById("start").onclick=start;
document.getElementById("stop").onclick=stop;
document.getElementById("export").onclick=()=>{
 const csv="Time,Event\n"+events.map(e=>
  '"'+e.time.replaceAll('"','""')+'","'+e.event.replaceAll('"','""')+'"'
 ).join("\n");
 const a=document.createElement("a");
 a.href=URL.createObjectURL(new Blob([csv],{type:"text/csv"}));
 a.download="drivesafe-events.csv";a.click();
 URL.revokeObjectURL(a.href);
};
</script>
</body>
</html>
"""

@app.route("/")
def home():
    return render_template_string(PAGE)

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8501, debug=False)



Overwriting /content/app.py


In [44]:
import subprocess, time, requests, os

# Stop any previous process listening on port 8501
subprocess.run(
    "fuser -k 8501/tcp",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(2)

log = open("/content/drivesafe.log", "w")
process = subprocess.Popen(
    ["python", "-u", "/content/app.py"],
    stdout=log,
    stderr=subprocess.STDOUT
)

for _ in range(20):
    try:
        r = requests.get("http://127.0.0.1:8501", timeout=2)
        print("Server status:", r.status_code)
        print("DriveSafe AI page loaded:", "Live camera monitoring" in r.text)
        break
    except requests.RequestException:
        time.sleep(1)
else:
    print(open("/content/drivesafe.log").read())

Server status: 200
DriveSafe AI page loaded: True


In [45]:
from getpass import getpass
from pyngrok import ngrok

token = getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(token)

tunnel = ngrok.connect(8501, "http")
print("Your DriveSafe AI website:", tunnel.public_url)

Enter your ngrok authtoken: ··········
Your DriveSafe AI website: https://entire-showbiz-deputize.ngrok-free.dev


In [46]:
import os

app_path = "/content/app.py"

print("App file exists:", os.path.isfile(app_path))

if os.path.isfile(app_path):
    print("File size:", os.path.getsize(app_path), "bytes")
    with open(app_path, "r", encoding="utf-8") as f:
        code = f.read()

    print("Has webcam code:", "getUserMedia" in code)
    print("Has eye detection:", "FaceLandmarker" in code)
    print("Has alarm code:", "function alarm" in code)
else:
    print("The application file is missing.")

App file exists: True
File size: 9212 bytes
Has webcam code: True
Has eye detection: True
Has alarm code: True


In [47]:
import requests

try:
    response = requests.get(
        "http://127.0.0.1:8501/",
        timeout=5
    )
    print("Flask HTTP status:", response.status_code)
    print("Page title present:", "DriveSafe AI" in response.text)
except requests.RequestException as error:
    print("Flask is not responding:", error)

Flask HTTP status: 200
Page title present: True


In [48]:
import subprocess
import time
import requests
import os

# First check whether the app is already responding
try:
    response = requests.get("http://127.0.0.1:8501/", timeout=3)
    print("Flask is already responding:", response.status_code)
except requests.RequestException:
    # Start the existing application without replacing its code
    log_file = open("/content/drivesafe.log", "w")
    process = subprocess.Popen(
        ["python", "-u", "/content/app.py"],
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

    for attempt in range(20):
        time.sleep(1)
        try:
            response = requests.get(
                "http://127.0.0.1:8501/", timeout=2
            )
            print("Flask HTTP status:", response.status_code)
            break
        except requests.RequestException:
            if process.poll() is not None:
                print("Flask stopped unexpectedly.")
                break
    else:
        print("Flask did not respond within 20 seconds.")

    if process.poll() is not None or response.status_code if False else False:
        pass

    if os.path.exists("/content/drivesafe.log"):
        print("\nServer log:")
        print(open("/content/drivesafe.log").read()[-3000:])

Flask is already responding: 200


In [50]:
from getpass import getpass
from pyngrok import ngrok

# Close old tunnels managed by this notebook
ngrok.kill()

token = getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(token)

tunnel = ngrok.connect(
    addr="http://127.0.0.1:8501",
    proto="http"
)

print("New DriveSafe AI website link:")
print(tunnel.public_url)

Enter your ngrok authtoken: ··········
New DriveSafe AI website link:
https://entire-showbiz-deputize.ngrok-free.dev
